In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import pandas as pd

path = "/kaggle/input/competitions/playground-series-s6e7"

train = pd.read_csv(f"{path}/train.csv")
test = pd.read_csv(f"{path}/test.csv")
sample_submission = pd.read_csv(f"{path}/sample_submission.csv")

print(train.shape)
print(test.shape)

train.head()

## 1. Inspect Dataset Structure
"train.info" to check the column names, data types, number of non-null values, and memory usage.

In [ ]:
train.info()

## 2. Check Target Distribution

We examine how many students belong to each health category. This helps us understand whether the classes are balanced.

In [ ]:
train["health_condition"].value_counts()

## 3. Check Missing Values

We count missing values in each column so we know which features may require cleaning or imputation.

In [ ]:
train.isnull().sum().sort_values(ascending=False)

## Initial EDA Conclusions

- The training dataset contains 690,088 rows and 15 columns.
- There are 13 predictive features after removing `id` and the target.
- The target is highly imbalanced: most students are `at-risk`, while `fit` is the smallest class.
- Several features contain missing values, especially `stress_level`, `sleep_duration`, and `sleep_quality`.
- Numerical and categorical features are both present, so preprocessing will be required.

## 4. Inspect Unique Values and Feature Types

We inspect the unique values in categorical columns and summary statistics for numerical columns.

In [ ]:
train.describe(include="all").T

## Feature Summary Conclusions

- The numerical feature ranges look reasonable; no obvious impossible values appear.
- All categorical features have only three categories, which should be easy to encode.
- Missing values appear across many features and must be handled.
- Because the target is strongly imbalanced, we should use balanced accuracy and stratified validation.
- The `id` column is only an identifier and should initially be excluded from training.

## 5. Prepare Features and Target

We separate the input features from the target column and remove the identifier column.

In [ ]:
X = train.drop(columns=["health_condition", "id"])
y = train["health_condition"]

X_test = test.drop(columns=["id"])

print(X.shape)
print(y.shape)
print(X_test.shape)

## 6. Create Training and Validation Sets

We split the labeled data into training and validation sets while preserving the class proportions.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(X_train.shape)
print(X_valid.shape)

## 7. Identify Numerical and Categorical Features

We separate numerical and categorical columns because they require different preprocessing.

In [ ]:
numeric_columns = X_train.select_dtypes(include=["number"]).columns.tolist()
categorical_columns = X_train.select_dtypes(exclude=["number"]).columns.tolist()

print("Numerical:", numeric_columns)
print("Categorical:", categorical_columns)

## 8. Build the Preprocessing Pipeline

We prepare numerical and categorical features so they can be used by a machine-learning model.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_columns),
    ("cat", categorical_pipeline, categorical_columns)
])

## 9. Train a Baseline Model

We combine the preprocessing steps with a Logistic Regression model and train it on the training data.

In [ ]:
from sklearn.linear_model import LogisticRegression

model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=1000,
        class_weight="balanced"
    ))
])

model.fit(X_train, y_train)

This trains the model while automatically cleaning and transforming the data first.

## 10. Evaluate our model on the validation set

In [ ]:
from sklearn.metrics import balanced_accuracy_score

y_pred = model.predict(X_valid)

score = balanced_accuracy_score(y_valid, y_pred)

print("Balanced Accuracy:", score)

## Baseline Result

The Logistic Regression baseline achieved a balanced accuracy of **0.8574** on the validation set. This is a strong starting point.

In [ ]:
#Next, inspect which classes it predicts well or poorly:
from sklearn.metrics import classification_report

print(classification_report(y_valid, y_pred))

## Classification Report Conclusion

The model identifies the minority classes well, with recall of 0.90 for `fit` and 0.89 for `unhealthy`. However, precision for these classes is low, meaning some `at-risk` students are incorrectly classified as minority classes.

## 11. Train on All Data and Create Submission

We retrain the full pipeline using all labeled training data, then predict the health condition for Kaggle's test set.

In [ ]:
model.fit(X, y)

test_predictions = model.predict(X_test)

submission = sample_submission.copy()
submission["health_condition"] = test_predictions

submission.to_csv("submission_logistic_regression.csv", index=False)

submission.head()